In [ ]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 12:10:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
import pandas as pd 

top_merchants = pd.read_csv("top_100_merchants.csv")
merch_ranking = pd.read_csv("merchant_segment_rankings_all.csv")   # note: "rankings", with the g

In [ ]:
import numpy as np

# Sort merchants by overall rank in ascending order and reset the index
merch_ranking = merch_ranking.sort_values('overall_rank').reset_index(drop=True)
top100_from_all = merch_ranking.head(100)

# Check whether each merchant ABN is unique across the dataset
print('abn unique:', merch_ranking.merchant_abn.is_unique) 

# Check for missing values in each column and display only columns containing missing values
nulls = merch_ranking.isna().sum()
print(nulls[nulls > 0].to_dict())  

abn unique: True
ranks contiguous: True
score sorted desc: True
top file == head(100): True
value = avg*count, max gap: 1.862645149230957e-09
customers > txns: 0
fraud_flag_observed_count == transaction_count: 1.0
2021-02-28 2022-10-26
{'std_transaction_value': 14, 'recent_month_growth': 310, 'name_merchant': 396, 'tags': 396, 'customer_mean_disadvantage_score': 2, 'customer_mean_economic_resources_score': 2, 'predicted_3m_revenue': 37, 'industry': 547, 'merchant_tier': 547, 'take_rate': 547}


In [11]:
import numpy as np, pandas as pd

merch_ranking = merch_ranking.sort_values('overall_rank').reset_index(drop=True)
merch_ranking['in_top100'] = merch_ranking.overall_rank <= 100
merch_ranking['value_share'] = merch_ranking.total_transaction_value / merch_ranking.total_transaction_value.sum()

In [ ]:
print(merch_ranking.shape)                   
missing = merch_ranking.isna().mean()
print(missing[missing > 0].round(3))              
print([c for c in merch_ranking.columns if merch_ranking[c].nunique() == 1]) 

(4422, 66)
std_transaction_value                     0.003
recent_month_growth                       0.070
name_merchant                             0.090
tags                                      0.090
customer_mean_disadvantage_score          0.000
customer_mean_economic_resources_score    0.000
predicted_3m_revenue                      0.008
industry                                  0.124
merchant_tier                             0.124
take_rate                                 0.124
dtype: float64
['fraud_flag_coverage', 'fraud_data_review_flag']


- There are 4,422 merchants in the file.
- Some information is missing for many merchants: Industry, tier and take rate are missing for about 12% of merchants, names and tags for 9%, and recent growth for 7%. 
- The "Unknown" segment in the later cells show the over lap between the Industry, tier and take rate and the names and tags. 

In [ ]:
# Produces a percentage of how much of the transaction value is held by best ranked merchants k 
# k=10: checks the top 10 ranked merchants 
# k=100: the top 100 ranked merchants relevant to our analysis 
# k=442: top 442 ranked merchants which is also the top 10% of our ranked merchants since we have 4422 merchants in total. 
# k=1000: the top 1000 ranked merchants 
for k in (10, 100, 442, 1000):
    print(k, round(merch_ranking.value_share.head(k).sum(), 3)) 

# Produces the percentage of how much of the transaction value is held by the top 100 merchants with purely transaction value (no ranking involved)
print('top 100 with transaction value only (no ranking)', round(merch_ranking.value_share.nlargest(100).sum(), 3)) 

10 0.035
100 0.27
442 0.508
1000 0.734
top 100 with transaction value only (no ranking) 0.319


- The 10 best-ranked merchants handle only 3.5% of all transaction value.
- The top 100 handle 27%, and the top 10% of merchants (442) handle about half (50.8%).
- The 1,000 best-ranked merchants handle about three quarters (73.4%).

So value is spread across many merchants, and picking 100 cannot capture most of the market.

If we simply chose the 100 merchants with the highest transaction value and ignored the ranking, they would handle 31.9% of the transaction value. The ranked top 100 give up about 5 percentage points of value because the ranking rewards other things too, such as repeat customers. The ranking is not just choosing the biggest merchants.

In [ ]:
top = merch_ranking[merch_ranking.in_top100]
rest = merch_ranking[~merch_ranking.in_top100]
similar = rest[rest.unique_customers >= top.unique_customers.min()]    # others as big as the top 100 (186 of them)

features = ['unique_customers', 'repeat_customer_share', 'avg_transaction_value', 'total_transaction_value',
            'monthly_value_cv', 'unique_customer_sa2s', 'take_rate']
print(pd.DataFrame({'top 100': top[features].median(),
                    'all others': rest[features].median(),
                    'others of similar size': similar[features].median()}).round(3))

                             top 100  all others  others of similar size
unique_customers           18228.500     350.000               11493.000
repeat_customer_share          0.542       0.008                   0.290
avg_transaction_value        150.622     355.568                  57.957
total_transaction_value  7148015.778  132952.089             1004346.887
monthly_value_cv               0.283       0.401                   0.283
unique_customer_sa2s        1487.000     245.000                1464.500
take_rate                      4.450       4.620                   4.845


The table compares the typical (median) top-100 merchant with all other merchants, and with the 186 other merchants who are as large in customer numbers as the smallest top-100 merchant. Comparing the top 100 merchants to other merchants of similaz sizes is the fairer comparison we use in this analysis. 

- **Repeat customers:** a typical top-100 merchant gets 54% of its business from returning customers, against 0.8% for other merchants and 29% for similar-sized ones. This stays a clear difference even among equally large merchants.
- **Purchase size and revenue:** top-100 merchants have a higher typical purchase ($151) than similar-sized merchants ($58). The total transaction value of the top 100 merchants is seven times the total transactions of the similar size merchants ($7.1M against $1.0M).
- **Steadiness and geographic reach:** these look better than for "all others", but they are the same as for similar-sized merchants (monthly variability 0.283 for both, and reach of about 1,490 against 1,460 districts). They come with being large and do not set the top 100 apart.
- **Take rate:** the top 100 pay the BNPL firm a slightly lower rate (4.45%) than similar-sized merchants (4.85%).

In [ ]:
# Produces the count of unique merchants, the total value share, the total number of merchants in the top 100, 
# the median transaction value and the average take rate for each of the 7 merchant segments 
by_segment = merch_ranking.groupby('segment').agg(merchants=('merchant_abn', 'count'), value_share=('value_share', 'sum'),
                                      top100=('in_top100', 'sum'), median_purchase_value=('avg_transaction_value', 'median'),
                                      take_rate=('take_rate', 'mean')).round(3)
print(by_segment.sort_values('value_share', ascending=False))

# Similarly produces total number of merchants, and the mean take rate for each of the merchant tiers of a, b, c
print(merch_ranking.groupby('merchant_tier').agg(merchants=('merchant_abn', 'count'), take_rate=('take_rate', 'mean')).round(2))

                        merchants  value_share  top100  median_purchase_value  \
segment                                                                         
Electronics & Digital        1135        0.243      26                203.641   
Fashion & Lifestyle          1083        0.232      28                218.496   
Arts, Media & Services        696        0.148      13                430.433   
Automotive & Outdoor          478        0.138      14                512.196   
Unknown                       547        0.112       7               1273.067   
Other                         332        0.081       8                562.591   
Home & Living                 151        0.045       4                320.145   

                        take_rate  
segment                            
Electronics & Digital       4.598  
Fashion & Lifestyle         4.530  
Arts, Media & Services      4.454  
Automotive & Outdoor        4.591  
Unknown                       NaN  
Other             

- Electronics & Digital (24.3% of value) and Fashion & Lifestyle (23.2%) are the largest segments, and together they supply 54 of the top 100.
- Home & Living is the smallest (151 merchants, 4.5% of value, 4 in the top 100).
- Typical purchase size differs a lot: about $200 in Electronics and Fashion against $430 to $560 in Arts, Automotive and Other.
- Take rates are similar across segments (about 4.4% to 4.7%).
- The "Unknown" segment (547 merchants, 11% of value) means missing information, not a type of merchant. It has no take rate, and the project asks for only 3 to 5 segments, so this needs resolving.

Merchants in tier **a** pay the BNPL firm the most (an average take rate of 6.23%), tier **b**  pays 4.09% and tier **c** pays 2.25%. The project documents do not say what the tier letters stand for, so describe them by their take rate.

In [ ]:
# Outputs the top 3 merchants with the highest recent monthly growth
print(merch_ranking.nlargest(3, 'recent_month_growth')[['merchant_abn', 'name_merchant', 'segment', 'overall_rank',
                                                  'previous_month_value', 'recent_month_value', 'recent_month_growth']].round(1))
# Outputs the number of merchants with only one customer
print((merch_ranking.unique_customers == 1).sum())

# Outputs the number of merchants where their single largest customer makes up more than half of their value 
print((merch_ranking.largest_customer_value_share > 0.5).sum())

# Outputs the median, 90th and 99th percentile of the average transaction value across merchants
print(merch_ranking.avg_transaction_value.quantile([.5, .9, .99]).round(0).to_dict())  

      merchant_abn                    name_merchant                 segment  \
2693   16496390781                              NaN                 Unknown   
4241   68889885215           Faucibus Lectus A Inc.  Arts, Media & Services   
4181   20415536115  Nibh Phasellus Nulla Industries           Home & Living   

      overall_rank  previous_month_value  recent_month_value  \
2693          2694                  30.4              8655.6   
4241          4242                  26.1               971.6   
4181          4182                 109.3              3878.4   

      recent_month_growth  
2693                283.7  
4241                 36.2  
4181                 34.5  
14
59
{0.5: 344.0, 0.9: 3141.0, 0.99: 17385.0}


- **Fastest growth:** the three fastest-growing merchants multiplied their monthly value by roughly 35 to 285 times, but started from very small amounts (for example, $30 to $8,656). Growth percentages from a tiny starting point are misleading, so a minimum starting value is needed before treating growth as a sign of success.
- **Reliance on one customer:** 14 merchants have only one customer, and 59 merchants get more than half of their value from a single customer. Their income depends on one person, which makes them risky.
- **Purchase sizes:** the typical merchant averages $344 per purchase, but the top 10% average over $3,141 and the top 1% over $17,385. Most merchants sell small purchases, with a small group selling very large ones.

In [ ]:
# Identifying the number of flagged merchants and how many times they have been flagged
flagged = merch_ranking[merch_ranking.any_flagged_transaction_count > 0]
print(len(flagged), flagged.any_flagged_transaction_count.value_counts().to_dict()) 

# Identifying the merchants with the high risk flag 
print(merch_ranking[merch_ranking.fraud_risk_flag][['overall_rank', 'transaction_count', 'fraud_rate_used']].round(3))

62 {1: 54, 2: 4, 3: 4}
      overall_rank  transaction_count  fraud_rate_used
2292          2293                 44            0.068
2982          2983                 11            0.091
3043          3044                 14            0.071
3065          3066                 16            0.062
3089          3090                  6            0.167
3363          3364                  3            0.333
4287          4288                  1            1.000
4295          4296                  1            1.000


- Fraud flags are rare: only 62 of the 4,422 merchants have any flagged purchase, and 54 of these have just one.
- 8 merchants are labelled high fraud risk. Each has only 1 to 44 purchases, so their fraud rates (6% up to 100%) come from very small numbers. Two merchants have a single purchase, and that one purchase was flagged, which gives a 100% rate.
- None of the high-risk merchants are near the top: the best-ranked high rish merchant is at 2,293.

A high fraud rate on a handful of purchases is weak evidence 

In [ ]:
big_base        = merch_ranking.unique_customers >= merch_ranking.unique_customers.quantile(.75)
small_base      = merch_ranking.unique_customers <= merch_ranking.unique_customers.quantile(.25)
large_purchases = merch_ranking.avg_transaction_value >= merch_ranking.avg_transaction_value.quantile(.75)

groups = {
    'Small customer base, large purchases': small_base & large_purchases,
    'Large customer base, some fraud flag': big_base & (merch_ranking.any_flagged_transaction_count > 0),
    'Low history':                          merch_ranking.low_history_flag,
}

rows = []
for name, condition in groups.items():
    members = merch_ranking[condition]
    rows.append({'group': name,
                 'merchants': len(members),
                 'in top 100': int(members.in_top100.sum()),
                 'median rank': members.overall_rank.median(),
                 'share of value (%)': round(members.value_share.sum() * 100, 1)})

group_summary = pd.DataFrame(rows).set_index('group')
print(group_summary)                                     

                                      merchants  in top 100  median rank  \
group                                                                      
Small customer base, large purchases        694           0       3531.5   
Large customer base, some fraud flag         33          11        171.0   
Low history                                 567           0       3808.0   

                                      share of value (%)  
group                                                     
Small customer base, large purchases                 3.2  
Large customer base, some fraud flag                 5.3  
Low history                                          1.2  


- **Few customers but large purchases:** 694 merchants. None are in the top 100, they handle only 3.2% of value, and they rank around 3,500th. They are a small niche that the ranking pushes down.
- **Many customers and some fraud flag:** 33 merchants, and 11 of them are in the top 100. They have only one or two flagged purchases among thousands of purchases, which is a very small share. A flag is not a reason to exclude a large merchant on its own.
- **Little history:** 567 merchants, none in the top 500, handling 1.2% of value. With so little history, their performance is hard to predict, so they could be reviewed again later.